# Lab 3 — Inspect & Talk to Models
**Day 1 Morning | ~45 minutes | Colab CPU**

---

## What You Will Build
By the end of this lab you will have:
1. Inspected a modern instruct model's architecture config (layers, heads, context window)
2. Compared greedy vs sampled generation and four temperature values
3. Seen exactly why **chat templates matter** — raw strings break instruct models
4. Built a stateful multi-turn `ChatSession` that tracks context growth
5. Debugged context-window pressure as a production concern

> **The key idea:** A model is not a text box. It has a contract — a specific token format, 
> a finite working memory, and sampling knobs that are product controls, not magic.

In [ ]:
!pip install -q transformers torch accelerate
print('Install complete')


In [ ]:
# Configuration — local model only, no API key needed
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'Config loaded — using {MODEL_ID}')


---

## Part A — Architecture Inspection (15 min)

We load `Qwen/Qwen2.5-0.5B-Instruct` — a modern, tiny instruct model (~1 GB on CPU). We are reading both the config and the actual module layout.

> First download takes 2–3 minutes. Subsequent runs are instant because Colab caches weights.

Expected shape: you should see roughly 0.5B parameters, dozens of transformer layers, and a context window printed in tokens. The exact values may change if the model repo updates, but the relative ideas stay the same.


In [ ]:
# Cell A1 — Load tokenizer and model config
# INSTRUCTOR NOTE: 'We load weights too so we can print the layer structure.
#                   The config alone would miss the actual parameter layout.'
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id  = MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16)
config    = model.config

attention_heads = config.num_attention_heads
kv_heads = getattr(config, 'num_key_value_heads', None)

print('KEY ARCHITECTURE NUMBERS')
print(f'  Model           : {model_id}')
print(f'  Hidden size     : {config.hidden_size}  (d_model)')
print(f'  Layers          : {config.num_hidden_layers}')
print(f'  Attention heads : {attention_heads}')
print(f'  KV heads (GQA)  : {kv_heads if kv_heads is not None else "N/A"}')
print(f'  Vocab size      : {config.vocab_size:,}')
print(f'  Context window  : {config.max_position_embeddings:,} tokens')

if kv_heads and kv_heads < attention_heads:
    ratio = attention_heads / kv_heads
    print()
    print('GQA CALLOUT')
    print(f'  This model uses fewer KV heads than attention heads ({attention_heads}:{kv_heads}, ratio {ratio:.1f}x).')
    print('  That is Grouped Query Attention: query heads share key/value heads.')
    print('  Deployment impact: smaller KV cache, lower memory pressure during long-context inference.')

params = sum(p.numel() for p in model.parameters())
print(f'\n  Parameters      : {params:,}  ({params/1e9:.2f}B)')
print(f'  Memory BF16     : ~{params*2/1e9:.2f} GB')
print(f'  Memory INT4     : ~{params*0.5/1e9:.2f} GB  ← what you will use in Lab 4')


In [ ]:
# Cell A2 — Layer structure: what is actually inside?
print('LAYER STRUCTURE (first 18 named modules)')
print(f'{"Name":<45} {"Type"}')
print('-' * 65)
for name, module in list(model.named_modules())[:18]:
    print(f'  {name:<43} {type(module).__name__}')
print('  ...')
print()
print('The repeating pattern (layers.0, layers.1, ...) is the transformer stack.')
print('Attention + MLP + LayerNorm — repeated num_hidden_layers times.')

---

## Part B — Generation Controls (15 min)

We generate from the local model. The point: understand how sampling settings
change output *before* you wire them into a production system.

In [ ]:
# Cell B1 — Helper: generate from local model
def generate(prompt_text, max_new_tokens=80, **kwargs):
    # If temperature is provided, sampling must be enabled or Transformers will warn/ignore it.
    if 'temperature' in kwargs and 'do_sample' not in kwargs:
        kwargs['do_sample'] = True

    msgs = [{'role': 'user', 'content': prompt_text}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs    = tokenizer(formatted, return_tensors='pt')
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id, **kwargs
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

q = 'What are the top 3 challenges in deploying LLMs to production? Be brief.'
print('GREEDY (deterministic) — run twice, expect identical output:')
print(f'Run 1: {generate(q, max_new_tokens=80, do_sample=False)[:120]}')
print(f'Run 2: {generate(q, max_new_tokens=80, do_sample=False)[:120]}')


In [ ]:
# Cell B2 — Temperature and top_p: production generation knobs
# INSTRUCTOR NOTE: ask: 'Which setting would you use for support answers? For brainstorming?'
test = 'List 3 steps to debug slow LLM inference in production:'
print('TEMPERATURE COMPARISON (same deployment prompt, four temperatures):\n')
for temp in [0.1, 0.5, 1.0, 1.5]:
    out = generate(test, max_new_tokens=55, temperature=temp)
    print(f'  temp={temp}: {out[:180]}\n')

print('=' * 70)
print('TOP-P COMPARISON (same temperature, narrower nucleus sampling):\n')
wide = generate(test, max_new_tokens=55, temperature=0.9, top_p=1.0)
narrow = generate(test, max_new_tokens=55, temperature=0.9, top_p=0.9)
print(f'  top_p=1.0: {wide[:180]}\n')
print(f'  top_p=0.9: {narrow[:180]}\n')
print('Deployment note: lower temperature/top_p is usually safer for support, tools, and structured output.')


In [ ]:
# Cell B3 — Why chat templates exist
# INSTRUCTOR NOTE: 'Raw prompting does not work on instruct models.
#                   The special tokens are part of the model contract.'
raw_prompt = 'What is quantization?'

formatted  = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': raw_prompt}],
    tokenize=False, add_generation_prompt=True
)

msgs_with_system = [
    {'role': 'system', 'content': 'You are an LLM deployment expert.'},
    {'role': 'user', 'content': raw_prompt},
]
formatted_with_system = tokenizer.apply_chat_template(
    msgs_with_system, tokenize=False, add_generation_prompt=True
)

print('Raw string sent to model:')
print(f'  {raw_prompt!r}')
print()
print('Formatted user-only chat template:')
print(repr(formatted))
print()
print('Formatted system + user chat template:')
print(repr(formatted_with_system))
print()
print('The im_start / im_end markers are the trained signal for instruction-following.')
print('System and user roles become different segments in the formatted prompt.')


---

## Part C — Multi-Turn Conversation (15 min)

For the chat section we keep using the local model. This preserves the key lesson: a chat app is just repeated requests plus explicit history management.

The `history` list below has the same structure as the `messages=[...]` list you pass to the OpenAI client. Local chat templates and hosted chat APIs share the same core protocol idea: every turn sends an ordered conversation transcript.


In [ ]:
# Cell C1 — ChatSession: the stateful conversation pattern
# INSTRUCTOR NOTE: 'Every production chatbot is basically this class with more error handling.'
class ChatSession:
    '''Tracks conversation history. Stateful chat = history accumulation.'''
    def __init__(self, system_prompt='You are a helpful assistant.'):
        self.history = [{'role': 'system', 'content': system_prompt}]

    def chat(self, user_message, max_new_tokens=90, warn_at=0.8):
        self.history.append({'role': 'user', 'content': user_message})
        used = self.token_estimate(add_generation_prompt=True)
        limit = config.max_position_embeddings
        if used / limit >= warn_at:
            print(f'WARNING: context is at {used/limit:.1%} of the model window ({used}/{limit} tokens).')

        formatted = tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(formatted, return_tensors='pt')
        input_len = inputs['input_ids'].shape[1]
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        answer = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        self.history.append({'role': 'assistant', 'content': answer})
        return answer

    def token_estimate(self, add_generation_prompt=False):
        '''Exact tokenizer count for the full formatted chat prompt.'''
        formatted = tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=add_generation_prompt
        )
        return len(tokenizer(formatted)['input_ids'])

    def show(self):
        icons = {'system': '[system]', 'user': '[user]', 'assistant': '[assistant]'}
        for m in self.history:
            preview = m['content'][:100] + ('...' if len(m['content']) > 100 else '')
            print(f"{icons[m['role']]} {m['role'].upper()}: {preview}")

    def show_full_prompt(self):
        '''Print the exact prompt string sent to the local model.'''
        print(tokenizer.apply_chat_template(
            self.history, tokenize=False, add_generation_prompt=True
        ))

    def trim(self, max_turns=3):
        '''Keep system prompt plus the last N user/assistant pairs.'''
        system = self.history[:1]
        turns = self.history[1:]
        keep_messages = max_turns * 2
        self.history = system + turns[-keep_messages:]


In [ ]:
# Cell C2 — Run a three-turn conversation
bot = ChatSession(
    system_prompt='You are an LLM deployment expert. Max 2 sentences per answer.'
)

print(bot.chat('What is quantization?'))
print(f'[Context: ~{bot.token_estimate()} tokens]\n')
print('─' * 40)
print(bot.chat('How does it compare to pruning?'))
print(f'[Context: ~{bot.token_estimate()} tokens]\n')
print('─' * 40)
print(bot.chat('Which should I try first when deploying a 7B model?'))
print(f'[Context: ~{bot.token_estimate()} tokens]\n')
print()
bot.show()

print('\nFULL PROMPT SENT TO THE MODEL:')
bot.show_full_prompt()


In [ ]:
# Cell C3 — Context window as a production constraint
# INSTRUCTOR NOTE: 'This is one of the most common silent production bugs.'
context_limit = config.max_position_embeddings
used          = bot.token_estimate()

print('CONTEXT WINDOW STATUS')
print(f'  Used      : ~{used:,} tokens')
print(f'  Limit     :  {context_limit:,} tokens  ({MODEL_ID})')
print(f'  Remaining : ~{context_limit - used:,} tokens')
print(f'  Pressure  :  {used/context_limit*100:.2f}%')
print()
print('When context fills in production:')
print('  Option A — Sliding window: drop oldest turns')
print('  Option B — Summarize:      compress old turns into a system prompt update')
print('  Option C — Retrieve:       keep old facts in a vector store and retrieve relevant pieces')
print('  Option D — New session:    carry a summary forward to a fresh history')
print()
print('Each option trades cost, latency, and continuity. There is no free lunch.')


---

## Required Exercise — Sliding Window Trim

Production chat systems cannot let history grow forever. Use the provided `trim(max_turns=N)` method to keep the system prompt plus the last N user/assistant pairs. Verify that the token estimate drops after trimming.


In [ ]:
print('Before trim:')
print(f'  messages: {len(bot.history)}')
print(f'  tokens  : {bot.token_estimate()}')

bot.trim(max_turns=1)

print('\nAfter trim to last 1 turn:')
print(f'  messages: {len(bot.history)}')
print(f'  tokens  : {bot.token_estimate()}')
print()
bot.show()


---

## ✅ Lab 3 Complete

You should now have:
- [ ] Architecture numbers for Qwen2.5-0.5B (hidden size, layers, context window)
- [ ] Greedy decoding: two identical outputs
- [ ] Temperature 0.1 is dry and consistent; 1.5 is creative and sometimes incoherent
- [ ] Chat template printed — raw string, user-only, and system+user formats
- [ ] Multi-turn conversation with context tracker
- [ ] Context window status printed
- [ ] Sliding-window trim exercise completed

## Stretch Goals

1. **Config comparison:** Load only the config (no weights) for `microsoft/phi-4` and `google/gemma-2-2b-it`.
   Compare their context windows and hidden sizes. Which would you choose for a 2K-token RAG prompt?
3. **Structured output:** Add `response_format={'type': 'json_object'}` to a chat call. 
   Ask the model to return its answer as JSON with keys `answer` and `confidence`. Parse and print.
3. **Top-k:** Modify the generate helper to accept `top_k`. Compare outputs at `top_k=20` vs `top_k=100` on the same deployment prompt.